# v6 FIX runs (GPU 0): continual + gentle per-step noise, branched from phase 60
**Question:** after phase 60 the continual run starts to break down (loss of plasticity). Does a gentle "perturb" step fix it?
* Branch: `v4b_continual_s0` phase 60 (Arabic) → phases 61, 62, ... (Hindi, Chinese, Arabic, ...), same recipe as before.
* Change: after every optimizer step, add Gaussian noise `σ × (each weight matrix's std at phase 60)` to every weight matrix. This is the "perturb" half of Shrink-and-Perturb (Dohare et al. 2024). The "shrink" half is already there: AdamW weight decay 0.1.
* Two noise levels side by side on the same GPU. Rough guide: σ per step ≈ (total noise per phase) / √(steps per phase ≈ 20,000). So 2e-4 ≈ 0.03 per phase (very gentle), 7e-4 ≈ 0.1 per phase (gentle).
* Health (weight norm, rarely-active neurons, effective rank) is printed at every eval and saved in `phases.jsonl`.
* Checkpoints `runs/<run>/ckpt/pXXX_<lang>.pt`, backed up to HF. Jobs run in the background: you can close the notebook.

In [1]:
TRAIN_LANGS = ["hi", "zh", "ar"]

import os, sys, json, glob, subprocess, time
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(); HF_USER = api.whoami()["name"]; HF_REPO = HF_USER + "/cml-lang-self-checkpoints"
print("code files:", {f: os.path.exists(f) for f in ["common_v6.py", "train_v6.py", "tokenizer_v4"]})

# FLORES+ dev + devtest (training evaluates on devtest). Skips files already here.
codes = {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}
os.makedirs("data/flores", exist_ok=True)
for l, code in codes.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            from datasets import load_dataset
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
            print("downloaded", p, len(ds))

# training data: data/en-{hi,zh,ar}/{train,val}.jsonl
LANGS_NEEDED = TRAIN_LANGS
need = [f"en-{l}/train.jsonl" for l in LANGS_NEEDED] + [f"en-{l}/val.jsonl" for l in ["hi", "zh", "ar"]]   # val of all 3: forgetting is measured
missing = [f"data/{n}" for n in need if not os.path.exists(f"data/{n}")]
if missing:   # try the HF dataset backup (only exists if you uploaded it once, see push_data_v6.py)
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(HF_USER + "/cml-lang-self-data", repo_type="dataset", local_dir="data",
                          allow_patterns=need)
    except Exception as e:
        print("no HF data backup:", type(e).__name__)
    missing = [m for m in missing if not os.path.exists(m)]
print("training data OK" if not missing else f"MISSING {missing}: copy data/ from the old machine or laptop (see push_data_v6.py)")

code files: {'common_v6.py': True, 'train_v6.py': True, 'tokenizer_v4': True}
training data OK


In [2]:
# the checkpoint we branch from (skips if already here)
SRC_RUN, SRC_PHASE = "v4b_continual_s0", 60
have = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")
if not have:
    name = next(f for f in api.list_repo_files(HF_REPO) if f.startswith(f"{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_"))
    hf_hub_download(HF_REPO, name, local_dir="runs")
    have = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")
INIT = have[0]; print("branch from:", INIT, f"{os.path.getsize(INIT) / 1e6:.0f} MB")

branch from: runs/v4b_continual_s0/ckpt/p060_ar.pt 267 MB


In [3]:
GPU = "0"
N_PHASES = 90                     # last phase; whatever is done by the morning gets analysed
NOISE = [2e-4, 7e-4]              # <- per-step noise levels (x starting std). Adjust if wanted.
JOBS = {f"v6_cont_noise{s:g}_from60": ["--mode", "continual", "--noise_per_step", str(s)] for s in NOISE}
print(JOBS)

{'v6_cont_noise0.0002_from60': ['--mode', 'continual', '--noise_per_step', '0.0002'], 'v6_cont_noise0.0007_from60': ['--mode', 'continual', '--noise_per_step', '0.0007']}


In [4]:
def launch(run, args):
    os.makedirs("logs", exist_ok=True)
    log = open(f"logs/{run}.out", "a")
    cmd = [sys.executable, "-u", "train_v6.py", "--gpu", GPU, "--run", run, "--init", INIT,
           "--start_phase", str(SRC_PHASE), "--n_phases", str(N_PHASES)] + args
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)  # survives the kernel
    print(f"started {run} (pid {p.pid}) -> logs/{run}.out"); return p

procs = {run: launch(run, a) for run, a in JOBS.items()}

started v6_cont_noise0.0002_from60 (pid 5653) -> logs/v6_cont_noise0.0002_from60.out
started v6_cont_noise0.0007_from60 (pid 5654) -> logs/v6_cont_noise0.0007_from60.out


In [6]:
# check progress any time (safe to rerun; does not touch the jobs)
for run in JOBS:
    print("=" * 30, run); os.system(f"tail -n 4 logs/{run}.out")
    f = f"runs/{run}/phases.jsonl"
    if os.path.exists(f):
        R = [json.loads(l) for l in open(f)]
        print("  phases done:", [r["phase"] for r in R][-6:], "| last chrF:", R[-1].get("chrf_masked"), "| health:", R[-1].get("health"))
os.system("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv")

============================== v6_cont_noise0.0002_from60
Validating                    : ██████████| 100%            
  ...02_from60/ckpt/p060_ar.pt: 100%|██████████|  267MB /  267MB            
  [p61 hi] step  1,000 | lr 2.0e-04 | train 2.833 | val 1.624 (best 1.624) | window Δ n/a | stable | 76k tok/s | phase 3m17s | run ETA: after first phase | health {'wnorm': 228.888, 'rank_enc1': 1079.2, 'rank_dec1': 679.4, 'rarely_active': 0.1489}
  [p61 hi] step  1,500 | lr 2.0e-04 | train 2.654 | val 1.511 (best 1.511) | window Δ n/a | stable | 77k tok/s | phase 4m52s | run ETA: after first phase | health {'wnorm': 229.036, 'rank_enc1': 1081.5, 'rank_dec1': 704.4, 'rarely_active': 0.1482}
  phases done: [60] | last chrF: {'hi': 7.91, 'zh': 2.79, 'ar': 52.34} | health: None
============================== v6_cont_noise0.0007_from60
  File "/venv/main/lib/python3.12/site-packages/torch/nn/functional.py", line 2993, in layer_norm
    return torch.layer_norm(
           ^^^^^^^^^^^^^^^^^
torch.Ou

0

In [ ]:
# to stop both jobs (only if needed). Rerunning the launch cell later RESUMES from the last saved phase.
# for run in JOBS: os.system(f"pkill -f 'train_v6.py.*--run {run}'")